# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mashelesc/flyrank_ml_internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Setup (run once at the top)
- Load the data-contract analysis table.
- Keep only rows with a defined `recovered` label (declining pages that have outcome-window data).

In [1]:
import duckdb
import numpy as np
import pandas as pd
from pathlib import Path

OUT = Path("work/outputs")
OUT.mkdir(parents = True, exist_ok = True)
path_str = str((OUT / "analysis_table.parquet").resolve()).replace("\\", "/")

con = duckdb.connect()
df = con.sql(f"""SELECT * FROM read_parquet('{path_str}')
                 WHERE recovered IS NOT NULL"""
            ).df()

print("Labeled rows:", len(df))
print(df["recovered"].value_counts())
print("Overall recovery rate:", round(float(df["recovered"].mean()), 4))

Labeled rows: 14299
recovered
1    8326
0    5973
Name: count, dtype: int64
Overall recovery rate: 0.5823


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*


**Rule name:** Baseline v0 — recovery priority for declining pages

**Idea:** Score each declining page with transparent points. Higher score =
higher priority to review / refresh for recovery.

**Point rule (feature window or static only — no outcome leakage):**

| Signal | Points | Reason code pattern |
|--------|--------|---------------------|
| position_tier 1-3 | +30 | pos_top3(+30) |
| position_tier 4-10 | +22 | pos_4_10(+22) |
| position_tier 11-20 | +12 | pos_11_20(+12) |
| position_tier 20+ | +4 | pos_deep(+4) |
| search_volume ≥ 1000 | +25 | volume_high(+25) |
| search_volume ≥ 100 | +15 | volume_mid(+15) |
| search_volume ≥ 10 | +8 | volume_low(+8) |
| search_volume &lt; 10 | +2 | volume_tiny(+2) |
| missing search_volume | +0 | volume_missing(+0) |
| content_age_days ≤ 180 | +20 | age_fresh(+20) |
| age ≤ 365 | +12 | age_1y(+12) |
| age ≤ 730 | +6 | age_2y(+6) |
| age &gt; 730 | +2 | age_old(+2) |
| keyword_token_count 3–8 | +10 | len_balanced(+10) |
| token count &gt; 8 | +5 | len_long(+5) |
| token count &lt; 3 | +3 | len_short(+3) |
| editorial content_type | +8 | type_editorial(+8) |
| commercial content_type | +6 | type_commercial(+6) |
| other type | +3 | type_other(+3) |
| impressions_feature ≥ 500 | +5 | imp_feature_strong(+5) |

**Outputs of this section:** `baseline_score`, `baseline_reasons`  
- Each reason code records which branch fired so a human can read why a page ranked high or low.

In [2]:
def baseline_score_row(r: pd.Series) -> tuple[float, list[str]]:
    """Transparent recovery-priority score + reason codes."""
    score = 0.0
    reasons = []

    tier = r.get("position_tier")
    if tier == "1-3":
        score += 30; reasons.append("pos_top3(+30)")
    elif tier == "4-10":
        score += 22; reasons.append("pos_4_10(+22)")
    elif tier == "11-20":
        score += 12; reasons.append("pos_11_20(+12)")
    elif tier == "20+":
        score += 4; reasons.append("pos_deep(+4)")
    else:
        reasons.append("pos_unknown(+0)")

    sv = r.get("search_volume")
    if pd.isna(sv):
        reasons.append("volume_missing(+0)")
    elif sv >= 1000:
        score += 25; reasons.append("volume_high(+25)")
    elif sv >= 100:
        score += 15; reasons.append("volume_mid(+15)")
    elif sv >= 10:
        score += 8; reasons.append("volume_low(+8)")
    else:
        score += 2; reasons.append("volume_tiny(+2)")

    age = r.get("content_age_days")
    if pd.isna(age):
        reasons.append("age_missing(+0)")
    elif age <= 180:
        score += 20; reasons.append("age_fresh(+20)")
    elif age <= 365:
        score += 12; reasons.append("age_1y(+12)")
    elif age <= 730:
        score += 6; reasons.append("age_2y(+6)")
    else:
        score += 2; reasons.append("age_old(+2)")

    tok = r.get("keyword_token_count")
    if pd.isna(tok):
        reasons.append("len_missing(+0)")
    elif 3 <= tok <= 8:
        score += 10; reasons.append("len_balanced(+10)")
    elif tok > 8:
        score += 5; reasons.append("len_long(+5)")
    else:
        score += 3; reasons.append("len_short(+3)")

    ctype = str(r.get("content_type") or "").lower()
    if ctype in {"blog", "article", "guide", "post"}:
        score += 8; reasons.append("type_editorial(+8)")
    elif ctype in {"product", "category", "landing"}:
        score += 6; reasons.append("type_commercial(+6)")
    elif ctype and ctype != "nan":
        score += 3; reasons.append("type_other(+3)")
    else:
        reasons.append("type_missing(+0)")

    imp = r.get("impressions_feature")
    if pd.notna(imp) and imp >= 500:
        score += 5; reasons.append("imp_feature_strong(+5)")

    return score, reasons

In [3]:
scores, reason_lists = [], []
for _, row in df.iterrows():
    s, rs = baseline_score_row(row)
    scores.append(s)
    reason_lists.append("|".join(rs))

df = df.copy()
df["baseline_score"] = scores
df["baseline_reasons"] = reason_lists

print("=== Section 1: rule applied ===")
print(df["baseline_score"].describe())
print("\nExample reason codes (5 rows):")
print(df[["baseline_score", "recovered", "position_tier", "baseline_reasons"]].head().to_string(index = False))

=== Section 1: rule applied ===
count    14299.000000
mean        52.950976
std         11.088855
min         20.000000
25%         45.000000
50%         54.000000
75%         61.000000
max         88.000000
Name: baseline_score, dtype: float64

Example reason codes (5 rows):
 baseline_score  recovered position_tier                                                                                  baseline_reasons
           39.0          1           20+                              pos_deep(+4)|volume_mid(+15)|age_1y(+12)|len_long(+5)|type_other(+3)
           57.0          1           1-3      pos_top3(+30)|volume_tiny(+2)|age_1y(+12)|len_long(+5)|type_other(+3)|imp_feature_strong(+5)
           54.0          1          4-10 pos_4_10(+22)|volume_tiny(+2)|age_1y(+12)|len_balanced(+10)|type_other(+3)|imp_feature_strong(+5)
           54.0          0          4-10 pos_4_10(+22)|volume_tiny(+2)|age_1y(+12)|len_balanced(+10)|type_other(+3)|imp_feature_strong(+5)
           54.0          1  

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*


**Steps:**
1. Rank pages by `baseline_score` (highest first).
2. Attach `baseline_rank` (1 = highest priority).
3. Write `work/outputs/baseline_recovery_queue.csv`.

**Queue columns:** ids, score, rank, reason codes, label, and key signals
used by the rule.

**Use:** This CSV is the transparent baseline ranking a model must try to beat.


In [4]:
df["baseline_rank"] = (df["baseline_score"].rank(ascending = False, method = "first").astype(int))
queue = df.sort_values("baseline_rank").copy()

export_cols = [column for column in ["content_hash_id", "client_hash_id", "baseline_score", "baseline_rank", "baseline_reasons", "recovered", "position_tier",
                           "search_volume", "content_age_days", "keyword_token_count", "content_type", "impressions_feature","avg_position_feature",]
               if column in queue.columns]

out_csv = OUT / "baseline_recovery_queue.csv"
queue[export_cols].to_csv(out_csv, index = False)

print("=== Section 2: ranked queue ===")
print("Rows written:", len(queue))
print("Wrote:", out_csv.resolve())
print("\nTop 10:")
print(queue.nsmallest(10, "baseline_rank")[[column for column in ["baseline_rank", "baseline_score", "recovered",
                                                                  "position_tier", "search_volume", "baseline_reasons"] if column in queue.columns]].to_string(index = False))
print("\nBottom 5:")
print(queue.nlargest(5, "baseline_rank")[[column for column in ["baseline_rank", "baseline_score", "recovered",
                                                                "position_tier", "baseline_reasons"] if column in queue.columns]].to_string(index = False))

=== Section 2: ranked queue ===
Rows written: 14299
Wrote: /home/sc/Videos/Courses/FlyRank/ML/flyrank_ml_internship/work/notebooks/work/outputs/baseline_recovery_queue.csv

Top 10:
 baseline_rank  baseline_score  recovered position_tier  search_volume                                                                                      baseline_reasons
             1            88.0          1           1-3          40500                        pos_top3(+30)|volume_high(+25)|age_fresh(+20)|len_balanced(+10)|type_other(+3)
             2            88.0          1           1-3           1600      pos_top3(+30)|volume_high(+25)|age_fresh(+20)|len_long(+5)|type_other(+3)|imp_feature_strong(+5)
             3            88.0          1           1-3           2400                        pos_top3(+30)|volume_high(+25)|age_fresh(+20)|len_balanced(+10)|type_other(+3)
             4            85.0          0           1-3           9900    pos_top3(+30)|volume_high(+25)|age_1y(+12)|len_balanc

## 3. Top-20 review

For the **20 highest** `baseline_score` pages, record:

| Field | Meaning |
|-------|---------|
| **action** | What a human should do next (review / refresh / monitor) |
| **reason codes** | Why the rule scored it high (`baseline_reasons`) |
| **confidence note** | How much we trust this pick given data gaps |
| **what would make it wrong** | How this recommendation could fail |

In [5]:
assert "baseline_rank" in df.columns, "Run Sections 1–2 first."

top20 = df.nsmallest(20, "baseline_rank").copy()

def suggest_action(r: pd.Series) -> str:
    tier = r.get("position_tier")
    sv = r.get("search_volume")
    age = r.get("content_age_days")
    # High visibility + demand → prioritize refresh/review
    if tier in ("1-3", "4-10") and pd.notna(sv) and sv >= 100:
        return "prioritize_refresh_review"
    if tier in ("1-3", "4-10"):
        return "review_then_refresh"
    if pd.notna(sv) and sv >= 1000:
        return "investigate_demand_drop"
    if pd.notna(age) and age > 730:
        return "consider_refresh_or_retire"
    return "monitor_and_spot_check"

def confidence_note(r: pd.Series) -> str:
    gaps = []
    if pd.isna(r.get("search_volume")):
        gaps.append("missing_search_volume")
    if pd.isna(r.get("content_age_days")):
        gaps.append("missing_age")
    if r.get("position_tier") in (None, "unknown"):
        gaps.append("unknown_position")
    if pd.isna(r.get("content_type")) or str(r.get("content_type")).lower() in ("", "nan"):
        gaps.append("missing_content_type")
    imp = r.get("impressions_feature")
    if pd.notna(imp) and imp < 100:
        gaps.append("low_feature_window_impressions")
    if not gaps:
        return "moderate–high: core signals present"
    return "lower: " + ", ".join(gaps)

def would_make_wrong(r: pd.Series) -> str:
    modes = [
        "recovery already underway for reasons the score ignores",
        "label noisy / short outcome window misclassified this page",
        "search_volume stale or not the true demand driver",
        "position tier is an average — volatile daily ranks differ",
        "client- or query-level shock (site issues, seasonality) not in features",
    ]
    # Page-specific emphasis
    extra = []
    if pd.isna(r.get("search_volume")):
        extra.append("demand assumed from other signals only")
    if r.get("position_tier") == "20+":
        extra.append("deep position may not be recoverable with refresh alone")
    if pd.notna(r.get("content_age_days")) and r.get("content_age_days") <= 90:
        extra.append("already fresh — decline may be non-content (tech/algo)")
    return "; ".join(extra + modes[:3])

rows = []
for _, r in top20.iterrows():
    rows.append({
        "baseline_rank": int(r["baseline_rank"]),
        "baseline_score": r["baseline_score"],
        "recovered": int(r["recovered"]) if pd.notna(r["recovered"]) else None,
        "action": suggest_action(r),
        "reason_codes": r.get("baseline_reasons", ""),
        "confidence_note": confidence_note(r),
        "what_would_make_it_wrong": would_make_wrong(r),
        "position_tier": r.get("position_tier"),
        "search_volume": r.get("search_volume"),
        "content_age_days": r.get("content_age_days"),
        "content_hash_id": r.get("content_hash_id"),
    })

top20_review = pd.DataFrame(rows)

print("=== TOP 20 REVIEW ===")
# Compact view for the notebook
display_cols = [
    "baseline_rank", "baseline_score", "recovered", "action",
    "confidence_note", "reason_codes",
]
print(top20_review[display_cols].to_string(index=False))

print("\n=== Failure modes (top 20) ===")
print(
    top20_review[["baseline_rank", "what_would_make_it_wrong"]]
    .to_string(index=False)
)

# export for the report
review_path = OUT / "baseline_top20_review.csv"
top20_review.to_csv(review_path, index=False)
print(f"\nWrote {review_path}")

print("\nSection 3 complete.")

=== TOP 20 REVIEW ===
 baseline_rank  baseline_score  recovered                    action                     confidence_note                                                                                          reason_codes
             1            88.0          1 prioritize_refresh_review moderate–high: core signals present                        pos_top3(+30)|volume_high(+25)|age_fresh(+20)|len_balanced(+10)|type_other(+3)
             2            88.0          1 prioritize_refresh_review moderate–high: core signals present      pos_top3(+30)|volume_high(+25)|age_fresh(+20)|len_long(+5)|type_other(+3)|imp_feature_strong(+5)
             3            88.0          1 prioritize_refresh_review moderate–high: core signals present                        pos_top3(+30)|volume_high(+25)|age_fresh(+20)|len_balanced(+10)|type_other(+3)
             4            85.0          0 prioritize_refresh_review moderate–high: core signals present    pos_top3(+30)|volume_high(+25)|age_1y(+12)|len_

## 4. Weak picks + leakage check
### Weak picks
Inspect the **bottom** of the queue and any **top** picks that look inconsistent
(e.g. high score but weak signals, or recovered=0 with max points).

Ask: which picks look wrong and why?

Common “looks wrong” patterns:
- High score but missing volume / unknown position (score driven by thin priors)
- High score + very fresh content (decline may not be a content-staleness problem)
- Low score but `recovered = 1` (rule missed a real recovery path)
- Extreme scores from a single factor

### Leakage check
Confirm the baseline **does not** use:
- product decision / health / priority flags
- outcome-window metrics as score inputs
- `recovered` itself as an input
- any feature that requires dates after the feature window (after 2026-03-31)

Score inputs allowed: `position_tier`, `search_volume`, `content_age_days`,
`keyword_token_count`, `content_type`, `impressions_feature` (feature window only).

In [6]:
# 4a. Weak / suspicious picks
bottom15 = df.nlargest(15, "baseline_rank").copy()
top_misses = (df.nsmallest(50, "baseline_rank").query("recovered == 0").head(10))   # Top picks that recovered = 0
bottom_hits = (df.nlargest(50, "baseline_rank").query("recovered == 1").head(10))   # Bottom picks that recovered = 1

# High score but thin evidence
thin = df[(df["baseline_score"] >= df["baseline_score"].quantile(0.9)) & (df["search_volume"].isna() 
                                                                          | (df["position_tier"].isin(["unknown", None])))].nsmallest(10, "baseline_rank")

weak_cols = [column for column in ["baseline_rank", "baseline_score", "recovered", "position_tier", "search_volume", "content_age_days", "baseline_reasons",] if column in df.columns]

print("=== BOTTOM 15 (lowest priority) ===")
print(bottom15[weak_cols].to_string(index = False))

print("\n=== TOP-ish misses (high rank, recovered=0) ===")
if len(top_misses):
    print(top_misses[weak_cols].to_string(index = False))
else:
    print("(none in top 50)")

print("\n=== BOTTOM-ish hits (low rank, recovered=1) ===")
if len(bottom_hits):
    print(bottom_hits[weak_cols].to_string(index = False))
else:
    print("(none in bottom 50)")

print("\n=== High score + thin evidence (missing volume or unknown pos) ===")
if len(thin):
    print(thin[weak_cols].to_string(index = False))
else:
    print("(none)")

# 4b. Leakage check
print("\n" + "=" * 60)
print("LEAKAGE CHECK")
print("=" * 60)

# Columns used by baseline_score_row (must match Section 1)
ALLOWED_SCORE_INPUTS = {"position_tier", "search_volume", "content_age_days", "keyword_token_count", "content_type","impressions_feature",}

# Fields that must NOT influence the score
FORBIDDEN_NAME_BITS = ["outcome", "recover", "label", "target", "decision", "priority", "health", "refresh_score", "product", "gsc_clicks",]
score_related = [column for column in df.columns if any(x in column.lower() for x in ["baseline", "score", "reason", "rank"])]
print("Score-related columns present:", score_related)

# recovered must not be in the scoring function inputs
print("\n[ ] recovered used as score input?  NO (computed after features; not in ALLOWED_SCORE_INPUTS)")

# Product-like flags in frame
product_like = [column for column in df.columns if any(x in column.lower() for x in ["decision", "priority", "health", "product_score", "refresh_flag", "action_flag",])]
print("Product-like columns in df:", product_like if product_like else "(none)")
print("Used in baseline rule? NO — rule only reads ALLOWED_SCORE_INPUTS")

# Outcome / future-looking columns may exist for evaluation only
outcome_like = [column for column in df.columns if any(x in ccolumn.lower() for x in ["outcome", "recover"])]
print("Outcome/label columns in df (OK for eval, not for scoring):", outcome_like)

# Hard assert: mean score should differ from a pure leak of the label
# (if score were built from recovered, corr would be ~1)
corr = df["baseline_score"].corr(df["recovered"])
print(f"\nCorr(baseline_score, recovered) = {corr:.4f}")
print("  (expect modest, not ~1.0; high ~1.0 would suggest label leakage into score)")

# List what the rule actually reads
print("\nAllowed score inputs:")
for column in sorted(ALLOWED_SCORE_INPUTS):
    present = column in df.columns
    print(f"  {column}: {'present' if present else 'MISSING'}")

# Confirm impressions_feature is not outcome
if "impressions_outcome" in df.columns:
    print("\nimpressions_outcome present in table: YES (label-side / analysis only)")
    print("  baseline rule uses impressions_feature only — not impressions_outcome")

=== BOTTOM 15 (lowest priority) ===
 baseline_rank  baseline_score  recovered position_tier  search_volume  content_age_days                                                                               baseline_reasons
         14299            20.0          0           20+              0               432                            pos_deep(+4)|volume_tiny(+2)|age_2y(+6)|len_long(+5)|type_other(+3)
         14298            20.0          0           20+              0               390                            pos_deep(+4)|volume_tiny(+2)|age_2y(+6)|len_long(+5)|type_other(+3)
         14297            21.0          1           20+           <NA>               375 pos_deep(+4)|volume_missing(+0)|age_2y(+6)|len_short(+3)|type_other(+3)|imp_feature_strong(+5)
         14296            22.0          1           20+           <NA>               253                       pos_deep(+4)|volume_missing(+0)|age_1y(+12)|len_short(+3)|type_other(+3)
         14295            22.0          0   

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.